#### Lets create Temp View of Silver tables that we want. 
- Using SQL over Pyspark is same from Execution POV as both use Catlayst optimizer. 
- Its choice based on what we want to do. 
- If string relted transfomrations , multiple file (parquests , delta's) are to be processed then we geerally use PySpark. 
- When EDA , querying data is needed we use SQL. 
- But again its a choice of Proahrammer .

Temp views can only be created on dataframes or using read function on table as hown below , as createTmpView is method. and it can be called only on dF

In [0]:

spark.read.table('ecommerce.silver.slv_brands').createOrReplaceTempView("tmp_brands");
spark.read.table('ecommerce.silver.slv_products').createOrReplaceTempView("tmp_products");
spark.read.table('ecommerce.silver.slv_category').createOrReplaceTempView("tmp_category");

In [0]:
%sql
select * from tmp_category  limit 10 

In [0]:
%sql
create or replace table ecommerce.gold.dim_products as 
with cte_brands as 
(
  select b.brand_code,b.brand_name, c.category_code, c.category_name 
  from 
    tmp_brands as b INNER JOIN tmp_category as c 
    ON b.category_code = c.category_code
)
select
  p.*,
  coalesce(cte.brand_name, 'NA') as brand_name,
  coalesce(cte.category_name, 'NA') as category_name
from
  tmp_products as p LEFT JOIN cte_brands cte 
on
  p.category_code = cte.category_code and
  p.brand_code = cte.brand_code;

  



#### Very interesting bot below . 
- for customers we have cities and states listed , but imagine where we also want to pupulate Region for this rows . 
- so imagine we got this mapping from business and we want to add it as seperate column in customers table.

In [0]:
spark.read.table("ecommerce.silver.slv_customers").createOrReplaceTempView("tmp_customers");


In [0]:
%sql
select * from tmp_customers limit 10 

In [0]:
%sql
select distinct state from tmp_customers where country = 'Singapore'

### Lets create mapping 
 - Country -> region -> State

In [0]:
ind_region = {
    "west" : ['MH','GJ','RJ'],
    "south" : ['KA','TN','TS','AP','KL'],
    "north" : ['UP','WB','DL']
}
,
aus_region = {
    'south' : ['VIC'],
    'west' : ['WA'],
    'east' : ['NSW'],
    'north' : ['QLD']
}
,
uk_region = {
    'England' : ['ENG'],
    'Wales' : ['WLS'],
    'Scotland' : ['SCT'],
    'Northren Ireland' : ['NIR']
}
,
us_region = {
    'north' : ['MA', 'NY'],
    'south' : ['FL', 'TX'],
    'east' : ['NJ'],
    'west' : ['CA']
}
,

# UAE states
uae_region = {
    'west' : "AUH",
    "east": ["Dubai"], 
    "SHJ": ["Sharjah"]
}
,
# Singapore states
singapore_region = {
    'west' : "SG"
}
,
# Canada states
canada_region = {
    'west' : ['BC','AB'],
    'east' : ['ON','QC','NS','IL']
}
,
region_mapping = {
    'India' : ind_region,
    'Australia' : aus_region,
    'United Kingdom' : uk_region,
    'United States' : us_region,
    'United Arab Emirates' : uae_region,
    'Singapore' : singapore_region,
    'Canada' : canada_region
}




##### Now we have to covert above data mapping to dataframe like structure so we can join it with customer table and add region column :)
- Python Row function is used to create such structure 


In [0]:
from pyspark.sql import Row

data_mapping = []
for country_name , country_mapping  in region_mapping.items(): 
    for country , region in country_mapping.items(): #ind_region 
        for each_abbr in region:
            for each_short in region:
                data_mapping.append(Row(country= country_name,
                                        region=country, 
                                        state= each_short)
                )

            

data_mapping[:4]

In [0]:
demp = [Row(id=1,name='s'),
        Row(id=2,name='p')]
df = spark.createDataFrame(demp)

### Now that we have row like above we can create dataframe out of it . 


In [0]:
df_region_mapping = spark.createDataFrame(data_mapping)
df_region_mapping.show(5)

In [0]:
spark.read.table('ecommerce.silver.slv_customers').createOrReplaceTempView('tmp_customers')

### Lets add region column to customers table and write to gold .
- just so you know - converting tmp_view to 
dataframe - df = spark.table('tmp_view_name')
- also see how using just **df = spark.sql('')**
we can simply assign output from tmp_tables to dataframe

In [0]:
# join region df to custoer df 
# convert region df to tmp_view.

df_region_mapping.createOrReplaceTempView('tmp_df_region_mapping')


In [0]:
%python

df_customers = spark.sql("""
select 
    c.*,
    rm.region 
from 
    tmp_customers c left join 
    tmp_df_region_mapping rm
    on c.country = rm.country and c.state = rm.state
    """
)


#### Check count by each region

In [0]:
from pyspark.sql.functions import col
df_customers.groupBy(col('region')).count().show()



#### Now write it as External delta + gold table 

In [0]:

# Write final df to gold as Dim_customers 
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://sj-dbr-demo-proj/gold_data/dim_customers")

spark.sql(
    """
    create table if not exists  ecommerce.gold.dim_customers
    using delta 
    location 's3://sj-dbr-demo-proj/gold_data/dim_customers'
    """
)

In [0]:
dbutils.notebook.exit('SUCCESS')